# NB4 · Ratios financieros, DuPont y explicación de variaciones

## Objetivo

Calcular ratios conectados con la toma de decisiones y preparar su explicación en Power BI.

Este notebook:

- integra Balance y PyG;
- calcula rentabilidad, liquidez, eficiencia y solvencia;
- construye el análisis DuPont;
- calcula variaciones mensuales, anuales y YTD;
- asigna estados y alertas;
- prepara una tabla larga para Power BI.

> No se busca calcular muchos ratios, sino ratios que expliquen cómo evoluciona el negocio.

In [1]:
# ==========================================================
# 1. LIBRERÍAS Y RUTAS
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", 150)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.4f}"
)


# ==========================================================
# RUTAS DEL PROYECTO
# ==========================================================

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
elif (CWD.parent / "data").exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Debe existir una carpeta 'data/' en la raíz."
    )


PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / "04"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUTAS = {
    "balance": (
        PROCESSED_DIR
        / "03"
        / "03_balance_mensual.csv"
    ),

    "control_balance": (
        PROCESSED_DIR
        / "03"
        / "03_control_cuadre_balance.csv"
    ),

    "pyg": (
        PROCESSED_DIR
        / "03"
        / "03_magnitudes_pyg.csv"
    ),
}


# ==========================================================
# COMPROBACIÓN
# ==========================================================

for nombre, ruta in RUTAS.items():

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró '{ruta.name}'. "
            "Ejecuta primero el NB3."
        )


print(f"Raíz del proyecto: {BASE_DIR}")
print(f"Salida NB4:         {OUTPUT_DIR}")

print("\nArchivos encontrados:")

for nombre, ruta in RUTAS.items():
    print(f" - {nombre}: {ruta.name}")

Raíz del proyecto: C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios
Salida NB4:         C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\processed\04

Archivos encontrados:
 - balance: 03_balance_mensual.csv
 - control_balance: 03_control_cuadre_balance.csv
 - pyg: 03_magnitudes_pyg.csv


In [2]:
# ==========================================================
# 2. CARGA Y VALIDACIÓN DE DATOS
# ==========================================================

# ==========================================================
# COLUMNAS NECESARIAS
# ==========================================================

COLUMNAS_BALANCE = [
    "periodo",
    "masa_financiera",
    "categoria_financiera",
    "subcategoria_financiera",
    "importe_balance",
]

COLUMNAS_CONTROL = [
    "periodo",
    "estado",
    "diferencia",
]

COLUMNAS_PYG = [
    "periodo",
    "ingresos_explotacion",
    "aprovisionamientos",
    "margen_bruto",
    "gastos_personal",
    "otros_gastos_operativos",
    "ebitda",
    "amortizaciones",
    "deterioros",
    "ebit",
    "resultado_financiero",
    "resultado_no_recurrente",
    "resultado_antes_impuestos",
    "impuesto_beneficios",
    "resultado_neto",
    "ingresos_explotacion_ytd",
    "aprovisionamientos_ytd",
    "margen_bruto_ytd",
    "gastos_personal_ytd",
    "otros_gastos_operativos_ytd",
    "ebitda_ytd",
    "amortizaciones_ytd",
    "deterioros_ytd",
    "ebit_ytd",
    "resultado_financiero_ytd",
    "resultado_no_recurrente_ytd",
    "resultado_antes_impuestos_ytd",
    "impuesto_beneficios_ytd",
    "resultado_neto_ytd",
]


# ==========================================================
# VALIDACIÓN DE COLUMNAS
# ==========================================================

def validar_columnas_csv(ruta, esperadas):

    disponibles = pd.read_csv(
        ruta,
        nrows=0
    ).columns.tolist()

    faltantes = [
        columna
        for columna in esperadas
        if columna not in disponibles
    ]

    if faltantes:
        raise ValueError(
            f"El archivo '{ruta.name}' no contiene "
            f"las columnas necesarias: {faltantes}"
        )


validar_columnas_csv(
    RUTAS["balance"],
    COLUMNAS_BALANCE
)

validar_columnas_csv(
    RUTAS["control_balance"],
    COLUMNAS_CONTROL
)

validar_columnas_csv(
    RUTAS["pyg"],
    COLUMNAS_PYG
)


# ==========================================================
# CARGA
# ==========================================================

balance = pd.read_csv(
    RUTAS["balance"],
    parse_dates=["periodo"]
)

control_balance = pd.read_csv(
    RUTAS["control_balance"],
    parse_dates=["periodo"]
)

pyg = pd.read_csv(
    RUTAS["pyg"],
    parse_dates=["periodo"]
)


# ==========================================================
# CONTROLES BÁSICOS
# ==========================================================

if balance.empty:
    raise ValueError(
        "03_balance_mensual.csv está vacío."
    )

if pyg.empty:
    raise ValueError(
        "03_magnitudes_pyg.csv está vacío."
    )

if control_balance.empty:
    raise ValueError(
        "03_control_cuadre_balance.csv está vacío."
    )


# ==========================================================
# PERIODOS
# ==========================================================

periodos_balance = pd.DatetimeIndex(
    balance["periodo"]
    .drop_duplicates()
    .sort_values()
)

periodos_pyg = pd.DatetimeIndex(
    pyg["periodo"]
    .drop_duplicates()
    .sort_values()
)

periodos_control = pd.DatetimeIndex(
    control_balance["periodo"]
    .drop_duplicates()
    .sort_values()
)


if not periodos_balance.equals(periodos_pyg):
    raise ValueError(
        "Balance y PyG no contienen exactamente "
        "los mismos periodos."
    )

if not periodos_balance.equals(periodos_control):
    raise ValueError(
        "Balance y control de cuadre no contienen "
        "exactamente los mismos periodos."
    )


# ==========================================================
# CONTINUIDAD MENSUAL
# ==========================================================

periodos_esperados = pd.date_range(
    start=periodos_balance.min(),
    end=periodos_balance.max(),
    freq="MS"
)

periodos_faltantes = (
    periodos_esperados
    .difference(periodos_balance)
)

if len(periodos_faltantes) > 0:
    raise ValueError(
        "Faltan meses en NB3: "
        f"{[
            p.strftime('%Y-%m')
            for p in periodos_faltantes
        ]}"
    )

In [3]:
# ==========================================================
# 3. FUNCIONES AUXILIARES
# ==========================================================

TOLERANCIA_DIVISION = 0.000001


def dividir(numerador, denominador):
    """
    División segura.
    Permite denominadores positivos o negativos,
    pero no cero o prácticamente cero.
    """

    return np.where(
        denominador.abs()
        > TOLERANCIA_DIVISION,
        numerador / denominador,
        np.nan
    )


def dividir_positivo(numerador, denominador):
    """
    División para ratios cuya interpretación requiere
    denominador estrictamente positivo.
    """

    return np.where(
        denominador
        > TOLERANCIA_DIVISION,
        numerador / denominador,
        np.nan
    )


def obtener_serie(df, columna, valor):
    """
    Obtiene una serie mensual del Balance.
    """

    return (
        df.loc[
            df[columna].eq(valor)
        ]
        .groupby(
            "periodo",
            observed=True
        )[
            "importe_balance"
        ]
        .sum()
    )


def variacion_porcentual(actual, anterior):
    """
    Variación porcentual segura.
    """

    return np.where(
        anterior.abs()
        > TOLERANCIA_DIVISION,
        actual / anterior - 1,
        np.nan
    )

In [4]:
# ==========================================================
# 4. MAGNITUDES DEL BALANCE
# ==========================================================

periodos = pd.Index(
    sorted(
        balance[
            "periodo"
        ].unique()
    ),
    name="periodo"
)

finanzas = pd.DataFrame(
    index=periodos
)


# ==========================================================
# MASAS PRINCIPALES
# ==========================================================

mapa_masas = {
    "Activo corriente":
        "activo_corriente",

    "Activo no corriente":
        "activo_no_corriente",

    "Pasivo corriente":
        "pasivo_corriente",

    "Pasivo no corriente":
        "pasivo_no_corriente",

    "Patrimonio neto":
        "patrimonio_neto",
}


for masa, nombre in mapa_masas.items():

    finanzas[nombre] = (
        obtener_serie(
            balance,
            "masa_financiera",
            masa
        )
        .reindex(
            periodos,
            fill_value=0.0
        )
    )


# ==========================================================
# CATEGORÍAS
# ==========================================================

mapa_categorias = {
    "Tesorería":
        "tesoreria",

    "Existencias":
        "existencias",

    "Clientes y deudores":
        "clientes",

    "Proveedores y acreedores":
        "proveedores",

    "Deuda financiera":
        "deuda_financiera",
}

for categoria, nombre in mapa_categorias.items():

    finanzas[nombre] = (
        obtener_serie(
            balance,
            "categoria_financiera",
            categoria
        )
        .reindex(
            periodos,
            fill_value=0.0
        )
    )


# ==========================================================
# MAGNITUDES CALCULADAS
# ==========================================================

finanzas[
    "activo_total"
] = (
    finanzas[
        "activo_corriente"
    ]
    + finanzas[
        "activo_no_corriente"
    ]
)

finanzas[
    "pasivo_total"
] = (
    finanzas[
        "pasivo_corriente"
    ]
    + finanzas[
        "pasivo_no_corriente"
    ]
)

finanzas[
    "pasivo_pn_total"
] = (
    finanzas[
        "pasivo_total"
    ]
    + finanzas[
        "patrimonio_neto"
    ]
)


# ==========================================================
# DEUDA
# ==========================================================

finanzas[
    "deuda_financiera_bruta"
] = (
    finanzas[
        "deuda_financiera"
    ]
    .clip(lower=0)
)


finanzas[
    "deuda_financiera_neta"
] = (
    finanzas[
        "deuda_financiera_bruta"
    ]
    - finanzas[
        "tesoreria"
    ]
)


finanzas[
    "fondo_maniobra"
] = (
    finanzas[
        "activo_corriente"
    ]
    - finanzas[
        "pasivo_corriente"
    ]
)


# ==========================================================
# CONTROL DE CUADRE
# ==========================================================

finanzas[
    "diferencia_balance"
] = (
    finanzas[
        "activo_total"
    ]
    - finanzas[
        "pasivo_pn_total"
    ]
)


max_diferencia_balance = (
    finanzas[
        "diferencia_balance"
    ]
    .abs()
    .max()
)


if max_diferencia_balance > 0.01:
    raise ValueError(
        "Las magnitudes reconstruidas en NB4 "
        "no mantienen el cuadre del Balance."
    )


print(
    "Máxima diferencia Balance:",
    max_diferencia_balance
)

display(finanzas.head())

Máxima diferencia Balance: 3.725290298461914e-09


,activo_corriente,activo_no_corriente,pasivo_corriente,pasivo_no_corriente,patrimonio_neto,tesoreria,existencias,clientes,proveedores,deuda_financiera,activo_total,pasivo_total,pasivo_pn_total,deuda_financiera_bruta,deuda_financiera_neta,fondo_maniobra,diferencia_balance
periodo,,,,,,,,,,,,,,,,,
2021-01-01,"2,987,136.8800","21,322,881.7700","1,442,961.2700","23,282,366.7300","-415,309.3500","2,716,429.9700",0.0000,"83,348.1800","1,226,887.9400","23,278,079.2000","24,310,018.6500","24,725,328.0000","24,310,018.6500","23,278,079.2000","20,561,649.2300","1,544,175.6100",0.0000
2021-02-01,"3,077,633.1600","21,204,609.8100","1,536,525.7600","23,282,366.7300","-536,649.5200","3,005,062.8000",0.0000,"42,911.4800","1,327,038.1200","23,278,079.2000","24,282,242.9700","24,818,892.4900","24,282,242.9700","23,278,079.2000","20,273,016.4000","1,541,107.4000",0.0000
2021-03-01,"3,198,649.7800","21,098,306.4500","1,681,123.6400","23,282,366.7300","-666,534.1400","2,955,519.5400",0.0000,"59,594.0700","1,453,831.1500","23,278,079.2000","24,296,956.2300","24,963,490.3700","24,296,956.2300","23,278,079.2000","20,322,559.6600","1,517,526.1400",0.0000
2021-04-01,"2,780,444.8100","20,962,507.5100","1,022,943.8300","23,282,366.7300","-562,358.2400","2,530,975.3200",0.0000,"51,131.5200","896,840.1500","23,278,224.8900","23,742,952.3200","24,305,310.5600","23,742,952.3200","23,278,224.8900","20,747,249.5700","1,757,500.9800",0.0000
2021-05-01,"3,029,571.3800","20,806,279.3200","1,012,136.1900","23,282,366.7300","-458,652.2200","2,549,645.9800",0.0000,"32,964.7200","641,621.0200","23,279,757.0600","23,835,850.7000","24,294,502.9200","23,835,850.7000","23,279,757.0600","20,730,111.0800","2,017,435.1900",-0.0000


In [5]:
# ==========================================================
# 5. INTEGRACIÓN CON LA PYG
# ==========================================================

columnas_pyg = COLUMNAS_PYG.copy()


finanzas = (
    finanzas
    .reset_index()
    .merge(
        pyg[columnas_pyg],
        on="periodo",
        how="left",
        validate="one_to_one"
    )
    .sort_values("periodo")
    .reset_index(drop=True)
)


# ==========================================================
# CONTROL DE DATOS
# ==========================================================

if (
    finanzas[
        "ingresos_explotacion"
    ]
    .isna()
    .any()
):
    raise ValueError(
        "Existen meses de Balance sin PyG asociada."
    )


# ==========================================================
# VARIABLES TEMPORALES
# ==========================================================

finanzas["año"] = (
    finanzas[
        "periodo"
    ]
    .dt.year
    .astype("int16")
)

finanzas[
    "mes_numero"
] = (
    finanzas[
        "periodo"
    ]
    .dt.month
    .astype("int8")
)

finanzas[
    "año_mes"
] = (
    finanzas[
        "periodo"
    ]
    .dt.strftime("%Y-%m")
)


finanzas[
    "fecha_fin_mes"
] = (
    finanzas[
        "periodo"
    ]
    + pd.offsets.MonthEnd(0)
)


finanzas[
    "dias_transcurridos"
] = (
    finanzas[
        "fecha_fin_mes"
    ]
    .dt.dayofyear
)


finanzas[
    "dias_año"
] = np.where(
    finanzas[
        "periodo"
    ].dt.is_leap_year,
    366,
    365
)


finanzas[
    "factor_anualizacion"
] = (
    finanzas[
        "dias_año"
    ]
    / finanzas[
        "dias_transcurridos"
    ]
)


# ==========================================================
# MEDIAS YTD DE BALANCE
# ==========================================================

columnas_balance_medio = [
    "activo_total",
    "patrimonio_neto",
    "clientes",
    "proveedores",
    "existencias",
]


for columna in columnas_balance_medio:

    finanzas[
        f"{columna}_medio_ytd"
    ] = (
        finanzas
        .groupby(
            "año",
            observed=True
        )[columna]
        .expanding()
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
    )


# ==========================================================
# FLUJOS ANUALIZADOS
# ==========================================================

finanzas[
    "resultado_neto_anualizado"
] = (
    finanzas[
        "resultado_neto_ytd"
    ]
    * finanzas[
        "factor_anualizacion"
    ]
)


finanzas[
    "ingresos_anualizados"
] = (
    finanzas[
        "ingresos_explotacion_ytd"
    ]
    * finanzas[
        "factor_anualizacion"
    ]
)


# ==========================================================
# EBITDA LTM
# ==========================================================

finanzas[
    "ebitda_ltm"
] = (
    finanzas[
        "ebitda"
    ]
    .rolling(
        window=12,
        min_periods=12
    )
    .sum()
)


display(finanzas.tail())

,periodo,activo_corriente,activo_no_corriente,pasivo_corriente,pasivo_no_corriente,patrimonio_neto,tesoreria,existencias,clientes,proveedores,deuda_financiera,activo_total,pasivo_total,pasivo_pn_total,deuda_financiera_bruta,deuda_financiera_neta,fondo_maniobra,diferencia_balance,ingresos_explotacion,aprovisionamientos,margen_bruto,gastos_personal,otros_gastos_operativos,ebitda,amortizaciones,deterioros,ebit,resultado_financiero,resultado_no_recurrente,resultado_antes_impuestos,impuesto_beneficios,resultado_neto,ingresos_explotacion_ytd,aprovisionamientos_ytd,margen_bruto_ytd,gastos_personal_ytd,otros_gastos_operativos_ytd,ebitda_ytd,amortizaciones_ytd,deterioros_ytd,ebit_ytd,resultado_financiero_ytd,resultado_no_recurrente_ytd,resultado_antes_impuestos_ytd,impuesto_beneficios_ytd,resultado_neto_ytd,año,mes_numero,año_mes,fecha_fin_mes,dias_transcurridos,dias_año,factor_anualizacion,activo_total_medio_ytd,patrimonio_neto_medio_ytd,clientes_medio_ytd,proveedores_medio_ytd,existencias_medio_ytd,resultado_neto_anualizado,ingresos_anualizados,ebitda_ltm
31,2023-08-01,"6,156,899.8700","18,924,019.3600","2,887,783.9700","18,252,366.7300","3,940,768.5300","5,715,460.2300",0.0000,"36,335.8400","1,905,031.3000","18,278,217.0600","25,080,919.2300","21,140,150.7000","25,080,919.2300","18,278,217.0600","12,562,756.8300","3,269,115.9000",0.0000,"1,085,682.8100","-26,341.2800","1,059,341.5300","-271,522.9700","-592,456.1700","195,362.3900","-150,637.4000",0.0000,"44,724.9900",0.0000,0.0000,"44,724.9900","-11,181.2500","33,543.7400","9,883,827.9700","-221,920.6400","9,661,907.3300","-2,126,499.8300","-4,193,986.8000","3,341,420.7000","-1,206,153.7000",0.0000,"2,135,267.0000","668,106.9900","-5,551.5900","2,797,822.4000","-408,906.7000","2,388,915.7000",2023,8,2023-08,2023-08-31,243,365,1.5021,"25,616,362.7088","2,975,284.6775","52,299.5213","1,394,326.0237",0.0000,"3,588,289.0144","14,846,079.0496","5,197,446.9900"
32,2023-09-01,"5,675,796.7600","18,849,240.5300","2,344,770.3100","18,252,366.7300","3,927,900.2500","4,936,971.0500",0.0000,"63,020.9600","1,437,569.2900","18,280,371.7000","24,525,037.2900","20,597,137.0400","24,525,037.2900","18,280,371.7000","13,343,400.6500","3,331,026.4500",0.0000,"1,224,763.2900","-29,946.0800","1,194,817.2100","-271,917.6400","-559,067.4600","363,832.1100","-151,542.2700",0.0000,"212,289.8400","-218,508.7000","-10,938.8400","-17,157.7000","4,289.4200","-12,868.2800","11,108,591.2600","-251,866.7200","10,856,724.5400","-2,398,417.4700","-4,753,054.2600","3,705,252.8100","-1,357,695.9700",0.0000,"2,347,556.8400","449,598.2900","-16,490.4300","2,780,664.7000","-404,617.2800","2,376,047.4200",2023,9,2023-09,2023-09-30,273,365,1.3370,"25,495,104.3289","3,081,130.8522","53,490.7922","1,399,130.8311",0.0000,"3,176,766.6971","14,852,145.8238","5,352,286.0400"
33,2023-10-01,"5,471,808.9000","18,734,933.8600","1,938,039.5900","18,252,366.7300","4,016,336.4400","5,265,577.9800",0.0000,"52,795.1000","1,028,342.4300","18,280,371.7000","24,206,742.7600","20,190,406.3200","24,206,742.7600","18,280,371.7000","13,014,793.7200","3,533,769.3100",0.0000,"1,149,896.7200","-22,914.7100","1,126,982.0100","-281,203.9300","-568,543.3200","277,234.7600","-152,169.1800",0.0000,"125,065.5800","-1,750.0600","-5,400.6000","117,914.9200","-29,478.7300","88,436.1900","12,258,487.9800","-274,781.4300","11,983,706.5500","-2,679,621.4000","-5,321,597.5800","3,982,487.5700","-1,509,865.1500",0.0000,"2,472,622.4200","447,848.2300","-21,891.0300","2,898,579.6200","-434,096.0100","2,464,483.6100",2023,10,2023-10,2023-10-31,304,365,1.2007,"25,366,268.1720","3,174,651.4110","53,421.2230","1,362,051.9910",0.0000,"2,959,001.7028","14,718,250.3707","5,095,975.6000"
34,2023-11-01,"5,932,927.0200","18,612,690.5100","2,138,613.8400","18,252,366.7300","4,154,636.9600","5,703,230.6300",0.0000,"56,001.4100","1,135,291.7500","18,278,213.0600","24,545,617.5300","20,390,980.5700","24,545,617.5300","18,278,213.0600","12,574,982.4300","3,794,313.1800",0.0000,"1

In [6]:
# ==========================================================
# 6. RATIOS FINANCIEROS
# ==========================================================

ratios = finanzas[[
    "periodo",
    "año",
    "mes_numero",
    "año_mes",
]].copy()


# ==========================================================
# RENTABILIDAD MTD
# ==========================================================

ratios[
    "margen_bruto"
] = dividir(
    finanzas[
        "margen_bruto"
    ],
    finanzas[
        "ingresos_explotacion"
    ]
)


ratios[
    "margen_ebitda"
] = dividir(
    finanzas[
        "ebitda"
    ],
    finanzas[
        "ingresos_explotacion"
    ]
)


ratios[
    "margen_ebit"
] = dividir(
    finanzas[
        "ebit"
    ],
    finanzas[
        "ingresos_explotacion"
    ]
)


ratios[
    "margen_neto"
] = dividir(
    finanzas[
        "resultado_neto"
    ],
    finanzas[
        "ingresos_explotacion"
    ]
)


# ==========================================================
# RENTABILIDAD YTD
# ==========================================================

ratios[
    "margen_bruto_ytd"
] = dividir(
    finanzas[
        "margen_bruto_ytd"
    ],
    finanzas[
        "ingresos_explotacion_ytd"
    ]
)


ratios[
    "margen_ebitda_ytd"
] = dividir(
    finanzas[
        "ebitda_ytd"
    ],
    finanzas[
        "ingresos_explotacion_ytd"
    ]
)


ratios[
    "margen_ebit_ytd"
] = dividir(
    finanzas[
        "ebit_ytd"
    ],
    finanzas[
        "ingresos_explotacion_ytd"
    ]
)


ratios[
    "margen_neto_ytd"
] = dividir(
    finanzas[
        "resultado_neto_ytd"
    ],
    finanzas[
        "ingresos_explotacion_ytd"
    ]
)


# ==========================================================
# ROA / ROE YTD
# ==========================================================

ratios[
    "roa"
] = dividir_positivo(
    finanzas[
        "resultado_neto_anualizado"
    ],
    finanzas[
        "activo_total_medio_ytd"
    ]
)


ratios[
    "roe"
] = dividir_positivo(
    finanzas[
        "resultado_neto_anualizado"
    ],
    finanzas[
        "patrimonio_neto_medio_ytd"
    ]
)


# ==========================================================
# DUPONT
# ==========================================================

ratios[
    "dupont_margen_neto"
] = ratios[
    "margen_neto_ytd"
]


ratios[
    "dupont_rotacion_activos"
] = dividir_positivo(
    finanzas[
        "ingresos_anualizados"
    ],
    finanzas[
        "activo_total_medio_ytd"
    ]
)


ratios[
    "dupont_apalancamiento"
] = dividir_positivo(
    finanzas[
        "activo_total_medio_ytd"
    ],
    finanzas[
        "patrimonio_neto_medio_ytd"
    ]
)


ratios[
    "roe_dupont"
] = (
    ratios[
        "dupont_margen_neto"
    ]
    * ratios[
        "dupont_rotacion_activos"
    ]
    * ratios[
        "dupont_apalancamiento"
    ]
)


# ==========================================================
# LIQUIDEZ
# ==========================================================

ratios[
    "liquidez_corriente"
] = dividir_positivo(
    finanzas[
        "activo_corriente"
    ],
    finanzas[
        "pasivo_corriente"
    ]
)


ratios[
    "prueba_acida"
] = dividir_positivo(
    finanzas[
        "activo_corriente"
    ]
    - finanzas[
        "existencias"
    ],
    finanzas[
        "pasivo_corriente"
    ]
)


ratios[
    "liquidez_inmediata"
] = dividir_positivo(
    finanzas[
        "tesoreria"
    ],
    finanzas[
        "pasivo_corriente"
    ]
)


ratios[
    "fondo_maniobra_sobre_activo"
] = dividir_positivo(
    finanzas[
        "fondo_maniobra"
    ],
    finanzas[
        "activo_total"
    ]
)


# ==========================================================
# SOLVENCIA
# ==========================================================

ratios[
    "endeudamiento"
] = dividir_positivo(
    finanzas[
        "pasivo_total"
    ],
    finanzas[
        "activo_total"
    ]
)


ratios[
    "autonomia_financiera"
] = dividir_positivo(
    finanzas[
        "patrimonio_neto"
    ],
    finanzas[
        "activo_total"
    ]
)


ratios[
    "deuda_sobre_patrimonio"
] = dividir_positivo(
    finanzas[
        "deuda_financiera_bruta"
    ],
    finanzas[
        "patrimonio_neto"
    ]
)


ratios[
    "deuda_financiera_sobre_activo"
] = dividir_positivo(
    finanzas[
        "deuda_financiera_bruta"
    ],
    finanzas[
        "activo_total"
    ]
)


ratios[
    "peso_pasivo_corriente"
] = dividir_positivo(
    finanzas[
        "pasivo_corriente"
    ],
    finanzas[
        "pasivo_total"
    ]
)


ratios[
    "tesoreria_sobre_activo"
] = dividir_positivo(
    finanzas[
        "tesoreria"
    ],
    finanzas[
        "activo_total"
    ]
)


ratios[
    "deuda_neta_sobre_ebitda"
] = dividir_positivo(
    finanzas[
        "deuda_financiera_neta"
    ],
    finanzas[
        "ebitda_ltm"
    ]
)


# ==========================================================
# EFICIENCIA / CIRCULANTE
# ==========================================================

ratios[
    "rotacion_activos"
] = dividir_positivo(
    finanzas[
        "ingresos_anualizados"
    ],
    finanzas[
        "activo_total_medio_ytd"
    ]
)


ratios[
    "dias_cobro"
] = dividir_positivo(
    finanzas[
        "clientes_medio_ytd"
    ]
    * finanzas[
        "dias_transcurridos"
    ],
    finanzas[
        "ingresos_explotacion_ytd"
    ]
)


ratios[
    "existencias_sobre_activo"
] = dividir_positivo(
    finanzas[
        "existencias"
    ],
    finanzas[
        "activo_total"
    ]
)


# ==========================================================
# LIMPIEZA
# ==========================================================

ratios.replace(
    [
        np.inf,
        -np.inf,
    ],
    np.nan,
    inplace=True
)


# ==========================================================
# CONTROL DUPONT
# ==========================================================

control_dupont = ratios.loc[
    ratios["roe"].notna()
    & ratios["roe_dupont"].notna()
]


diferencia_dupont = (
    control_dupont[
        "roe"
    ]
    - control_dupont[
        "roe_dupont"
    ]
).abs()


if (
    not diferencia_dupont.empty
    and diferencia_dupont.max()
    > 0.000001
):
    raise ValueError(
        "La descomposición DuPont "
        "no reproduce el ROE."
    )


print(
    "Máxima diferencia ROE vs DuPont:",
    diferencia_dupont.max()
)

display(ratios.tail())

Máxima diferencia ROE vs DuPont: 2.220446049250313e-16


,periodo,año,mes_numero,año_mes,margen_bruto,margen_ebitda,margen_ebit,margen_neto,margen_bruto_ytd,margen_ebitda_ytd,margen_ebit_ytd,margen_neto_ytd,roa,roe,dupont_margen_neto,dupont_rotacion_activos,dupont_apalancamiento,roe_dupont,liquidez_corriente,prueba_acida,liquidez_inmediata,fondo_maniobra_sobre_activo,endeudamiento,autonomia_financiera,deuda_sobre_patrimonio,deuda_financiera_sobre_activo,peso_pasivo_corriente,tesoreria_sobre_activo,deuda_neta_sobre_ebitda,rotacion_activos,dias_cobro,existencias_sobre_activo
31,2023-08-01,2023,8,2023-08,0.9757,0.1799,0.0412,0.0309,0.9775,0.3381,0.2160,0.2417,0.1401,1.2060,0.2417,0.5796,8.6097,1.2060,2.1321,2.1321,1.9792,0.1303,0.8429,0.1571,4.6382,0.7288,0.1366,0.2279,2.4171,0.5796,1.2858,0.0000
32,2023-09-01,2023,9,2023-09,0.9755,0.2971,0.1733,-0.0105,0.9773,0.3335,0.2113,0.2139,0.1246,1.0310,0.2139,0.5825,8.2746,1.0310,2.4206,2.4206,2.1055,0.1358,0.8398,0.1602,4.6540,0.7454,0.1138,0.2013,2.4930,0.5825,1.3146,0.0000
33,2023-10-01,2023,10,2023-10,0.9801,0.2411,0.1088,0.0769,0.9776,0.3249,0.2017,0.2010,0.1167,0.9321,0.2010,0.5802,7.9903,0.9321,2.8234,2.8234,2.7170,0.1460,0.8341,0.1659,4.5515,0.7552,0.0960,0.2175,2.5539,0.5802,1.3248,0.0000
34,2023-11-01,2023,11,2023-11,0.9791,0.2771,0.1549,0.1158,0.9777,0.3206,0.1976,0.1935,0.1125,0.8715,0.1935,0.5813,7.7493,0.8715,2.7742,2.7742,2.6668,0.1546,0.8307,0.1693,4.3995,0.7447,0.1049,0.2324,2.6242,0.5813,1.3322,0.0000
35,2023-12-01,2023,12,2023-12,0.9511,0.3690,0.2192,0.0209,0.9759,0.3239,0.1990,0.1818,0.1044,0.7855,0.1818,0.5745,7.5217,0.7855,2.2929,2.2929,2.1727,0.1145,0.8203,0.1797,4.0760,0.7325,0.1080,0.1924,2.6839,0.5745,1.4274,0.0000


In [7]:
# ==========================================================
# 7. KPIs FINANCIEROS PARA POWER BI
# ==========================================================

metricas = []


def agregar_metrica(
    nombre,
    horizonte,
    serie
):
    """
    Añade una métrica al modelo largo de Power BI.
    """

    temp = finanzas[[
        "periodo",
        "año",
        "mes_numero",
        "año_mes",
    ]].copy()

    temp["variable"] = nombre
    temp["horizonte"] = horizonte
    temp["valor"] = serie

    metricas.append(temp)


# ==========================================================
# FLUJOS MTD
# ==========================================================

for nombre in [
    "ingresos_explotacion",
    "margen_bruto",
    "gastos_personal",
    "otros_gastos_operativos",
    "ebitda",
    "ebit",
    "resultado_neto",
]:

    agregar_metrica(
        nombre,
        "MTD",
        finanzas[nombre]
    )


# ==========================================================
# FLUJOS YTD
# ==========================================================

mapa_ytd = {
    "ingresos_explotacion":
        "ingresos_explotacion_ytd",

    "margen_bruto":
        "margen_bruto_ytd",

    "ebitda":
        "ebitda_ytd",

    "ebit":
        "ebit_ytd",

    "resultado_neto":
        "resultado_neto_ytd",
}


for nombre, columna in mapa_ytd.items():

    agregar_metrica(
        nombre,
        "YTD",
        finanzas[columna]
    )


# ==========================================================
# SALDOS DE CIERRE
# ==========================================================

columnas_cierre = [
    "activo_total",
    "activo_corriente",
    "activo_no_corriente",
    "pasivo_total",
    "pasivo_corriente",
    "pasivo_no_corriente",
    "patrimonio_neto",
    "tesoreria",
    "deuda_financiera_bruta",
    "deuda_financiera_neta",
    "clientes",
    "proveedores",
    "existencias",
    "fondo_maniobra",
]


for nombre in columnas_cierre:

    agregar_metrica(
        nombre,
        "CIERRE",
        finanzas[nombre]
    )


# ==========================================================
# UNIÓN
# ==========================================================

metricas_powerbi = pd.concat(
    metricas,
    ignore_index=True
)


# ==========================================================
# MISMO MES DEL AÑO ANTERIOR
# ==========================================================

comparativa_ly = (
    metricas_powerbi[[
        "periodo",
        "variable",
        "horizonte",
        "valor",
    ]]
    .copy()
)

comparativa_ly[
    "periodo"
] = (
    comparativa_ly[
        "periodo"
    ]
    + pd.DateOffset(years=1)
)

comparativa_ly = comparativa_ly.rename(
    columns={
        "valor":
        "valor_ly"
    }
)


metricas_powerbi = (
    metricas_powerbi
    .merge(
        comparativa_ly,
        on=[
            "periodo",
            "variable",
            "horizonte",
        ],
        how="left",
        validate="one_to_one"
    )
)


# ==========================================================
# VARIACIONES
# ==========================================================

metricas_powerbi[
    "variacion_abs"
] = (
    metricas_powerbi[
        "valor"
    ]
    - metricas_powerbi[
        "valor_ly"
    ]
)


metricas_powerbi[
    "variacion_pct"
] = variacion_porcentual(
    metricas_powerbi[
        "valor"
    ],
    metricas_powerbi[
        "valor_ly"
    ]
)


metricas_powerbi = (
    metricas_powerbi
    .sort_values(
        [
            "periodo",
            "variable",
            "horizonte",
        ]
    )
    .reset_index(drop=True)
)


display(
    metricas_powerbi.tail(20)
)

,periodo,año,mes_numero,año_mes,variable,horizonte,valor,valor_ly,variacion_abs,variacion_pct
916,2023-12-01,2023,12,2023-12,ebit,MTD,"214,737.2000","331,132.6700","-116,395.4700",-0.3515
917,2023-12-01,2023,12,2023-12,ebit,YTD,"2,872,385.3700","2,196,792.0900","675,593.2800",0.3075
918,2023-12-01,2023,12,2023-12,ebitda,MTD,"361,370.9300","478,481.4800","-117,110.5500",-0.2448
919,2023-12-01,2023,12,2023-12,ebitda,YTD,"4,674,804.3500","4,031,752.9400","643,051.4100",0.1595
920,2023-12-01,2023,12,2023-12,existencias,CIERRE,0.0000,0.0000,0.0000,NaN
921,2023-12-01,2023,12,2023-12,fondo_maniobra,CIERRE,"2,660,668.7100","2,502,316.5900","158,352.1200",0.0633
922,2023-12-01,2023,12,2023-12,gastos_personal,MTD,"-256,598.3500","-251,090.3000","-5,508.0500",0.0219
923,2023-12-01,2023,12,2023-12,ingresos_explotacion,MTD,"979,435.0200","963,983.6700","15,451.3500",0.0160
924,2023-12-01,2023,12,2023-12,ingresos_explotacion,YTD,"14,432,074.6000","13,512,643.8800","919,430.7200",0.0680
925,2023-12-01,2023,12,2023-12,margen_bruto,MTD,"931,523.1700","933,542.0000","-2,018.8300",-0.0022


In [8]:
# ==========================================================
# 8. RATIOS PARA POWER BI
# ==========================================================

ratios_powerbi = []

def agregar_ratio(
    nombre,
    horizonte,
    columna
):

    temp = ratios[[
        "periodo",
        "año",
        "mes_numero",
        "año_mes",
    ]].copy()

    temp["ratio"] = nombre
    temp["horizonte"] = horizonte
    temp["valor"] = ratios[columna]

    ratios_powerbi.append(temp)


# ==========================================================
# RATIOS MTD
# ==========================================================

for nombre in [
    "margen_bruto",
    "margen_ebitda",
    "margen_ebit",
    "margen_neto",
]:

    agregar_ratio(
        nombre,
        "MTD",
        nombre
    )


# ==========================================================
# RATIOS YTD
# ==========================================================

mapa_ratios_ytd = {
    "margen_bruto":
        "margen_bruto_ytd",

    "margen_ebitda":
        "margen_ebitda_ytd",

    "margen_ebit":
        "margen_ebit_ytd",

    "margen_neto":
        "margen_neto_ytd",

    "roa":
        "roa",

    "roe":
        "roe",

    "dupont_margen_neto":
        "dupont_margen_neto",

    "dupont_rotacion_activos":
        "dupont_rotacion_activos",

    "dupont_apalancamiento":
        "dupont_apalancamiento",

    "roe_dupont":
        "roe_dupont",

    "rotacion_activos":
        "rotacion_activos",

    "dias_cobro":
        "dias_cobro",
}


for nombre, columna in mapa_ratios_ytd.items():

    agregar_ratio(
        nombre,
        "YTD",
        columna
    )


# ==========================================================
# RATIOS DE CIERRE
# ==========================================================

ratios_cierre = [
    "liquidez_corriente",
    "prueba_acida",
    "liquidez_inmediata",
    "fondo_maniobra_sobre_activo",
    "endeudamiento",
    "autonomia_financiera",
    "deuda_sobre_patrimonio",
    "deuda_financiera_sobre_activo",
    "peso_pasivo_corriente",
    "tesoreria_sobre_activo",
    "existencias_sobre_activo",
]


for nombre in ratios_cierre:

    agregar_ratio(
        nombre,
        "CIERRE",
        nombre
    )


# ==========================================================
# LTM
# ==========================================================

agregar_ratio(
    "deuda_neta_sobre_ebitda",
    "LTM",
    "deuda_neta_sobre_ebitda"
)


# ==========================================================
# UNIÓN
# ==========================================================

ratios_largo = pd.concat(
    ratios_powerbi,
    ignore_index=True
)


# ==========================================================
# AÑO ANTERIOR
# ==========================================================

ratios_ly = (
    ratios_largo[[
        "periodo",
        "ratio",
        "horizonte",
        "valor",
    ]]
    .copy()
)

ratios_ly[
    "periodo"
] = (
    ratios_ly[
        "periodo"
    ]
    + pd.DateOffset(years=1)
)

ratios_ly = ratios_ly.rename(
    columns={
        "valor":
        "valor_año_anterior"
    }
)


ratios_largo = (
    ratios_largo
    .merge(
        ratios_ly,
        on=[
            "periodo",
            "ratio",
            "horizonte",
        ],
        how="left",
        validate="one_to_one"
    )
)


# ==========================================================
# VARIACIONES
# ==========================================================

ratios_largo[
    "variacion_anual"
] = (
    ratios_largo[
        "valor"
    ]
    - ratios_largo[
        "valor_año_anterior"
    ]
)


ratios_largo[
    "variacion_anual_pct"
] = variacion_porcentual(
    ratios_largo[
        "valor"
    ],
    ratios_largo[
        "valor_año_anterior"
    ]
)


ratios_largo = (
    ratios_largo
    .sort_values(
        [
            "periodo",
            "ratio",
            "horizonte",
        ]
    )
    .reset_index(drop=True)
)


display(ratios_largo.tail())

,periodo,año,mes_numero,año_mes,ratio,horizonte,valor,valor_año_anterior,variacion_anual,variacion_anual_pct
1003,2023-12-01,2023,12,2023-12,roa,YTD,0.1044,0.0341,0.0703,2.0582
1004,2023-12-01,2023,12,2023-12,roe,YTD,0.7855,0.7901,-0.0047,-0.0059
1005,2023-12-01,2023,12,2023-12,roe_dupont,YTD,0.7855,0.7901,-0.0047,-0.0059
1006,2023-12-01,2023,12,2023-12,rotacion_activos,YTD,0.5745,0.5446,0.0299,0.0549
1007,2023-12-01,2023,12,2023-12,tesoreria_sobre_activo,CIERRE,0.1924,0.1739,0.0186,0.1069


In [9]:
# ==========================================================
# 9. CATÁLOGO Y ESTADOS
# ==========================================================

catalogo_ratios = pd.DataFrame([

    # RENTABILIDAD
    ["margen_bruto", "Rentabilidad", "%", "Margen bruto / Ingresos",
     "Margen después de aprovisionamientos", "Mayor es mejor"],

    ["margen_ebitda", "Rentabilidad", "%", "EBITDA / Ingresos",
     "Rentabilidad operativa antes de amortizaciones", "Mayor es mejor"],

    ["margen_ebit", "Rentabilidad", "%", "EBIT / Ingresos",
     "Rentabilidad operativa después de amortizaciones", "Mayor es mejor"],

    ["margen_neto", "Rentabilidad", "%", "Resultado neto / Ingresos",
     "Beneficio final generado por las ventas", "Mayor es mejor"],

    ["roa", "Rentabilidad", "%", "Resultado YTD anualizado / Activo medio YTD",
     "Rentabilidad generada por los activos", "Mayor es mejor"],

    # DUPONT
    ["roe", "DuPont", "%", "Resultado YTD anualizado / Patrimonio medio YTD",
     "Rentabilidad de los recursos propios", "Mayor es mejor"],

    ["dupont_margen_neto", "DuPont", "%", "Resultado YTD / Ingresos YTD",
     "Componente de margen del ROE", "Mayor es mejor"],

    ["dupont_rotacion_activos", "DuPont", "Veces",
     "Ingresos YTD anualizados / Activo medio YTD",
     "Eficiencia en la utilización de activos", "Mayor es mejor"],

    ["dupont_apalancamiento", "DuPont", "Veces",
     "Activo medio YTD / Patrimonio medio YTD",
     "Componente de apalancamiento financiero",
     "Solo interpretable con patrimonio positivo"],

    ["roe_dupont", "DuPont", "%",
     "Margen × Rotación × Apalancamiento",
     "Reconstrucción del ROE mediante DuPont", "Mayor es mejor"],

    # LIQUIDEZ
    ["liquidez_corriente", "Liquidez", "Veces",
     "Activo corriente / Pasivo corriente",
     "Capacidad para atender obligaciones a corto plazo",
     "Vigilar niveles inferiores a 1"],

    ["prueba_acida", "Liquidez", "Veces",
     "(Activo corriente - Existencias) / Pasivo corriente",
     "Liquidez sin depender de inventarios",
     "Mayor que 1 suele ser favorable"],

    ["liquidez_inmediata", "Liquidez", "Veces",
     "Tesorería / Pasivo corriente",
     "Cobertura inmediata del pasivo corriente", "Depende del negocio"],

    ["fondo_maniobra_sobre_activo", "Liquidez", "%",
     "Fondo de maniobra / Activo",
     "Peso del colchón financiero a corto plazo", "Positivo es favorable"],

    # SOLVENCIA
    ["endeudamiento", "Solvencia", "%",
     "Pasivo / Activo",
     "Proporción del activo financiada externamente", "Menor es mejor"],

    ["autonomia_financiera", "Solvencia", "%",
     "Patrimonio neto / Activo",
     "Independencia respecto de financiación externa", "Mayor es mejor"],

    ["deuda_sobre_patrimonio", "Solvencia", "Veces",
     "Deuda financiera bruta / Patrimonio neto",
     "Presión de la deuda financiera sobre el patrimonio", "Menor es mejor"],

    ["deuda_financiera_sobre_activo", "Solvencia", "%",
     "Deuda financiera bruta / Activo",
     "Peso de la deuda financiera en la estructura", "Menor es mejor"],

    ["deuda_neta_sobre_ebitda", "Solvencia", "Veces",
     "Deuda financiera neta / EBITDA LTM",
     "Capacidad de repago mediante EBITDA", "Menor es mejor"],

    ["peso_pasivo_corriente", "Solvencia", "%",
     "Pasivo corriente / Pasivo total",
     "Peso de obligaciones de corto plazo",
     "Depende de la estructura financiera"],

    ["tesoreria_sobre_activo", "Solvencia", "%",
     "Tesorería / Activo",
     "Peso de liquidez disponible sobre el activo",
     "Depende del negocio"],

    # EFICIENCIA
    ["rotacion_activos", "Eficiencia", "Veces",
     "Ingresos anualizados / Activo medio YTD",
     "Ventas generadas por cada euro de activo", "Mayor es mejor"],

    # CIRCULANTE
    ["dias_cobro", "Circulante", "Días",
     "Clientes medios YTD × días transcurridos / Ingresos YTD",
     "Plazo medio aproximado de cobro", "Menor es mejor"],

    ["existencias_sobre_activo", "Circulante", "%",
     "Existencias / Activo",
     "Peso del inventario en la estructura financiera",
     "Depende del negocio"],

], columns=[
    "ratio",
    "grupo",
    "unidad",
    "formula",
    "interpretacion",
    "criterio",
])


# ==========================================================
# INTEGRACIÓN CON CATÁLOGO
# ==========================================================

ratios_largo = (
    ratios_largo
    .merge(
        catalogo_ratios,
        on="ratio",
        how="left",
        validate="many_to_one"
    )
)


# ==========================================================
# CONTROL DE CATÁLOGO
# ==========================================================

sin_catalogo = (
    ratios_largo[
        "grupo"
    ]
    .isna()
)

if sin_catalogo.any():

    ratios_error = (
        ratios_largo.loc[
            sin_catalogo,
            "ratio"
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "Existen ratios sin catálogo: "
        f"{ratios_error}"
    )


# ==========================================================
# ESTADOS
# ==========================================================

ratios_menor_mejor = {
    "endeudamiento",
    "deuda_sobre_patrimonio",
    "deuda_financiera_sobre_activo",
    "deuda_neta_sobre_ebitda",
    "dias_cobro",
}


ratios_informativos = {
    "liquidez_inmediata",
    "dupont_apalancamiento",
    "peso_pasivo_corriente",
    "tesoreria_sobre_activo",
    "existencias_sobre_activo",
}


def asignar_estado(fila):

    valor = fila["valor"]
    variacion = fila["variacion_anual"]
    ratio = fila["ratio"]

    if pd.isna(valor):
        return "SIN DATO"

    if ratio in ratios_informativos:
        return "INFORMATIVO"

    if ratio == "liquidez_corriente":

        if valor < 1:
            return "DESFAVORABLE"

        if valor <= 2.5:
            return "FAVORABLE"

        return "VIGILAR"

    if ratio == "prueba_acida":

        return (
            "FAVORABLE"
            if valor >= 1
            else "DESFAVORABLE"
        )

    if ratio == "fondo_maniobra_sobre_activo":

        return (
            "FAVORABLE"
            if valor >= 0
            else "DESFAVORABLE"
        )

    if ratio == "deuda_neta_sobre_ebitda":

        if valor < 0:
            return "FAVORABLE"

        if valor <= 3:
            return "FAVORABLE"

        if valor <= 4:
            return "VIGILAR"

        return "DESFAVORABLE"

    if pd.isna(variacion):
        return "SIN COMPARATIVA"

    if abs(variacion) < 0.0001:
        return "ESTABLE"

    if ratio in ratios_menor_mejor:

        return (
            "FAVORABLE"
            if variacion < 0
            else "DESFAVORABLE"
        )

    return (
        "FAVORABLE"
        if variacion > 0
        else "DESFAVORABLE"
    )


ratios_largo[
    "estado"
] = (
    ratios_largo
    .apply(
        asignar_estado,
        axis=1
    )
)


print(
    f"Ratios catalogados: "
    f"{len(catalogo_ratios)}"
)

display(ratios_largo.head())

Ratios catalogados: 24


,periodo,año,mes_numero,año_mes,ratio,horizonte,valor,valor_año_anterior,variacion_anual,variacion_anual_pct,grupo,unidad,formula,interpretacion,criterio,estado
0,2021-01-01,2021,1,2021-01,autonomia_financiera,CIERRE,-0.0171,NaN,NaN,NaN,Solvencia,%,Patrimonio neto / Activo,Independencia respecto de financiación externa,Mayor es mejor,SIN COMPARATIVA
1,2021-01-01,2021,1,2021-01,deuda_financiera_sobre_activo,CIERRE,0.9576,NaN,NaN,NaN,Solvencia,%,Deuda financiera bruta / Activo,Peso de la deuda financiera en la estructura,Menor es mejor,SIN COMPARATIVA
2,2021-01-01,2021,1,2021-01,deuda_neta_sobre_ebitda,LTM,NaN,NaN,NaN,NaN,Solvencia,Veces,Deuda financiera neta / EBITDA LTM,Capacidad de repago mediante EBITDA,Menor es mejor,SIN DATO
3,2021-01-01,2021,1,2021-01,deuda_sobre_patrimonio,CIERRE,NaN,NaN,NaN,NaN,Solvencia,Veces,Deuda financiera bruta / Patrimonio neto,Presión de la deuda financiera sobre el patrim...,Menor es mejor,SIN DATO
4,2021-01-01,2021,1,2021-01,dias_cobro,YTD,3.2527,NaN,NaN,NaN,Circulante,Días,Clientes medios YTD × días transcurridos / Ing...,Plazo medio aproximado de cobro,Menor es mejor,SIN COMPARATIVA


In [10]:
# ==========================================================
# 10. EXPLICACIÓN DUPONT
# ==========================================================

dupont = ratios[[
    "periodo",
    "roe",
    "dupont_margen_neto",
    "dupont_rotacion_activos",
    "dupont_apalancamiento",
]].copy()


# ==========================================================
# AÑO ANTERIOR
# ==========================================================

columnas_dupont = [
    "roe",
    "dupont_margen_neto",
    "dupont_rotacion_activos",
    "dupont_apalancamiento",
]


for columna in columnas_dupont:

    dupont[f"{columna}_año_anterior"] = (
        dupont[columna]
        .shift(12)
    )


dupont["roe_variacion"] = (
    dupont["roe"]
    - dupont["roe_año_anterior"]
)


# ==========================================================
# CONTRIBUCIONES
# ==========================================================

m0 = dupont["dupont_margen_neto_año_anterior"]
r0 = dupont["dupont_rotacion_activos_año_anterior"]
a0 = dupont["dupont_apalancamiento_año_anterior"]

m1 = dupont["dupont_margen_neto"]
r1 = dupont["dupont_rotacion_activos"]
a1 = dupont["dupont_apalancamiento"]


dupont["efecto_margen"] = (
    (m1 - m0)
    * r0
    * a0
)

dupont["efecto_rotacion"] = (
    m1
    * (r1 - r0)
    * a0
)

dupont["efecto_apalancamiento"] = (
    m1
    * r1
    * (a1 - a0)
)


dupont["variacion_explicada"] = (
    dupont["efecto_margen"]
    + dupont["efecto_rotacion"]
    + dupont["efecto_apalancamiento"]
)


dupont["diferencia_descomposicion"] = (
    dupont["roe_variacion"]
    - dupont["variacion_explicada"]
)


# ==========================================================
# PRINCIPAL CAUSA
# ==========================================================

def causa_dupont(fila):

    if (
        pd.isna(fila["roe"])
        or pd.isna(fila["roe_año_anterior"])
    ):
        return "Sin comparativa válida"

    efectos = {
        "Margen neto":
            fila["efecto_margen"],

        "Rotación de activos":
            fila["efecto_rotacion"],

        "Apalancamiento":
            fila["efecto_apalancamiento"],
    }

    efectos = {
        nombre: valor
        for nombre, valor in efectos.items()
        if pd.notna(valor)
    }

    if not efectos:
        return "Sin comparativa válida"

    return max(
        efectos,
        key=lambda nombre: abs(efectos[nombre])
    )


dupont["principal_causa_variacion_roe"] = (
    dupont.apply(
        causa_dupont,
        axis=1
    )
)


# ==========================================================
# CONTROL
# ==========================================================

control_dupont = dupont.loc[
    dupont["roe_variacion"].notna()
]

max_diferencia = (
    control_dupont["diferencia_descomposicion"]
    .abs()
    .max()
)

print(
    "Máxima diferencia de descomposición:",
    max_diferencia
)

display(dupont.tail())

Máxima diferencia de descomposición: 1.1102230246251565e-15


,periodo,roe,dupont_margen_neto,dupont_rotacion_activos,dupont_apalancamiento,roe_año_anterior,dupont_margen_neto_año_anterior,dupont_rotacion_activos_año_anterior,dupont_apalancamiento_año_anterior,roe_variacion,efecto_margen,efecto_rotacion,efecto_apalancamiento,variacion_explicada,diferencia_descomposicion,principal_causa_variacion_roe
31,2023-08-01,1.2060,0.2417,0.5796,8.6097,0.4820,0.0364,0.5227,25.3338,0.7241,2.7186,0.3481,-2.3427,0.7241,0.0000,Margen neto
32,2023-09-01,1.0310,0.2139,0.5825,8.2746,0.2059,0.0153,0.5240,25.7654,0.8251,2.6817,0.3228,-2.1794,0.8251,-0.0000,Margen neto
33,2023-10-01,0.9321,0.2010,0.5802,7.9903,0.5385,0.0395,0.5392,25.2867,0.3935,2.2024,0.2088,-2.0177,0.3935,-0.0000,Margen neto
34,2023-11-01,0.8715,0.1935,0.5813,7.7493,0.8573,0.0643,0.5530,24.1081,0.0142,1.7220,0.1319,-1.8398,0.0142,-0.0000,Apalancamiento
35,2023-12-01,0.7855,0.1818,0.5745,7.5217,0.7901,0.0627,0.5446,23.1395,-0.0047,1.5005,0.1258,-1.6310,-0.0047,0.0000,Apalancamiento


In [11]:
# ==========================================================
# 11. LIMPIEZA FINAL Y EXPORTACIÓN
# ==========================================================

def limpiar_para_exportar(
    df,
    decimales=4
):

    salida = df.copy()

    columnas_numericas = (
        salida
        .select_dtypes(
            include=[np.number]
        )
        .columns
    )

    for columna in columnas_numericas:

        salida.loc[
            salida[columna].abs()
            < 0.000001,
            columna
        ] = 0.0

        salida[columna] = (
            salida[columna]
            .round(decimales)
        )

    return salida


# ==========================================================
# EXPORTACIONES PREPARADAS
# ==========================================================

base_financiera_export = (
    limpiar_para_exportar(
        finanzas,
        decimales=2
    )
)

ratios_export = (
    limpiar_para_exportar(
        ratios,
        decimales=4
    )
)

ratios_powerbi_export = (
    limpiar_para_exportar(
        ratios_largo,
        decimales=4
    )
)

metricas_powerbi_export = (
    limpiar_para_exportar(
        metricas_powerbi,
        decimales=4
    )
)

dupont_export = (
    limpiar_para_exportar(
        dupont,
        decimales=4
    )
)


# ==========================================================
# LIMPIEZA DE CONTROLES
# ==========================================================

if (
    "diferencia_balance"
    in base_financiera_export.columns
):

    base_financiera_export.loc[
        base_financiera_export[
            "diferencia_balance"
        ].abs() <= 0.01,
        "diferencia_balance"
    ] = 0.0


if (
    "diferencia_descomposicion"
    in dupont_export.columns
):

    dupont_export.loc[
        dupont_export[
            "diferencia_descomposicion"
        ].abs() <= 0.0001,
        "diferencia_descomposicion"
    ] = 0.0


# ==========================================================
# EXPORTACIONES
# ==========================================================

EXPORTACIONES = {
    "04_ratios_mensual.csv":
        ratios_export,

    "04_ratios_powerbi.csv":
        ratios_powerbi_export,

    "04_metricas_powerbi.csv":
        metricas_powerbi_export,

    "04_catalogo_ratios.csv":
        catalogo_ratios,

    "04_dupont.csv":
        dupont_export,

    "04_base_financiera.csv":
        base_financiera_export,
}


for nombre, df in EXPORTACIONES.items():

    df.to_csv(
        OUTPUT_DIR / nombre,
        index=False,
        encoding="utf-8-sig",
        decimal="."
    )


print("Archivos exportados:")

for nombre in EXPORTACIONES:
    print(
        f" - data/processed/04/{nombre}"
    )


print("\n" + "=" * 55)
print("NB4 COMPLETADO CORRECTAMENTE")
print("=" * 55)

print(
    f"Periodo: "
    f"{finanzas['periodo'].min():%Y-%m} "
    f"→ "
    f"{finanzas['periodo'].max():%Y-%m}"
)

print(
    f"Meses procesados: "
    f"{finanzas['periodo'].nunique()}"
)

print(
    f"Ratios: "
    f"{len(catalogo_ratios)}"
)

print(
    f"Filas métricas Power BI: "
    f"{len(metricas_powerbi_export):,}"
)

print(
    f"Filas ratios Power BI: "
    f"{len(ratios_powerbi_export):,}"
)

print("=" * 55)

Archivos exportados:
 - data/processed/04/04_ratios_mensual.csv
 - data/processed/04/04_ratios_powerbi.csv
 - data/processed/04/04_metricas_powerbi.csv
 - data/processed/04/04_catalogo_ratios.csv
 - data/processed/04/04_dupont.csv
 - data/processed/04/04_base_financiera.csv

NB4 COMPLETADO CORRECTAMENTE
Periodo: 2021-01 → 2023-12
Meses procesados: 36
Ratios: 24
Filas métricas Power BI: 936
Filas ratios Power BI: 1,008


## Salidas principales

- `04_ratios_mensual.csv`: tabla ancha para análisis y forecast.
- `04_ratios_powerbi.csv`: tabla larga con valor, comparativas, estado y metadatos.
- `04_catalogo_ratios.csv`: fórmula, grupo, unidad e interpretación.
- `04_dupont.csv`: evolución del ROE y sus tres componentes.
- `04_base_financiera.csv`: magnitudes financieras utilizadas.

### Historia financiera

El ROE se explica mediante:

\[
ROE = Margen\ neto \times Rotación\ de\ activos \times Apalancamiento
\]

Después se comprueba si esa rentabilidad está acompañada por suficiente liquidez, eficiencia del circulante y una estructura de deuda sostenible.